# Secure Inference: HE vs MPC

Two approaches to running model inference on sensitive data **without revealing the input**.

| | HE (Homomorphic Encryption) | MPC (Multi-Party Computation) |
|---|---|---|
| **Library** | [TenSEAL](https://github.com/OpenMined/TenSEAL) (OpenMined) | [CrypTen](https://github.com/facebookresearch/CrypTen) (Meta) |
| **How it works** | Compute directly on ciphertext | Split data into secret shares across parties |
| **Parties needed** | 1 (single server) | 2+ (multi-party) |
| **ReLU / non-linear** | Polynomial approximation only | Full support |
| **Best for** | Small models (MLP, logistic regression) | Large models (DenseNet, BERT) |
| **Install** | `pip install tenseal` | `pip install crypten` |

This tutorial trains **one fraud detection model** and runs inference via **both** methods, comparing results.

---

## Contents

1. [Train a Plaintext Model](#1-train)
2. [HE: Encrypted Inference with TenSEAL](#2-he)
3. [MPC: Secret-Shared Inference](#3-mpc)
4. [Side-by-Side Comparison](#4-comparison)
5. [When to Use Which](#5-decision)

In [ ]:
import os, sys
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '../..'))
os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)

import torch
import torch.nn as nn
import numpy as np
import time

print(f'Working directory: {os.getcwd()}')

---
## 1. Train a Plaintext Model <a id='1-train'></a>

Train a fraud detection MLP on real data. This is the model both HE and MPC will use for encrypted inference.

In [ ]:
# Load real cancer survival data (METABRIC — 14 clinical features)
data = np.load('data/samples/cancer/data.npz')
X_all = torch.from_numpy(data['X'])
y_all = torch.from_numpy(data['y'])

# Train/test split
X_train, y_train = X_all[:1000], y_all[:1000]
X_test, y_test = X_all[1000:1020], y_all[1000:1020]  # 20 test patients

print(f'METABRIC cancer dataset: {len(X_all):,} patients, {X_all.shape[1]} features')
print(f'Features: age, tumor_size, lymph_nodes, grade, stage, chemo, hormone, radio, ER/HER2/PR status')
print(f'Target: overall survival (0=alive, 1=deceased)')
print(f'Death rate: {y_all.mean():.3f}')
print()

# Train a 3-layer MLP — deeper than fraud model to show HE limitations
model = nn.Sequential(
    nn.Linear(14, 32), nn.ReLU(),
    nn.Linear(32, 16), nn.ReLU(),
    nn.Linear(16, 1), nn.Sigmoid(),
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.BCELoss()

for epoch in range(100):
    pred = model(X_train).squeeze()
    loss = loss_fn(pred, y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

model.eval()
with torch.no_grad():
    preds = model(X_test).squeeze()
    acc = ((preds > 0.5).float() == y_test).float().mean()
    print(f'Model: {sum(p.numel() for p in model.parameters()):,} params (3 layers)')
    print(f'Plaintext accuracy: {acc:.3f} on {len(X_test)} test patients')
    print(f'Predictions: {[f"{p:.4f}" for p in preds[:5].tolist()]}')

---
## 2. HE: Encrypted Inference with TenSEAL <a id='2-he'></a>

With Homomorphic Encryption, the hospital encrypts patient data, sends ciphertext to the cloud, and the cloud runs the model **on encrypted data** without ever seeing the plaintext.

```
Hospital                          Cloud
   |                                |
   |  encrypt(patient)  ────────>   |
   |                         model(ciphertext)
   |  <──────── encrypt(prediction) |
   |                                |
   decrypt(prediction)              (never saw raw data)
```

### Limitation: HE only supports polynomial operations

CKKS supports addition and multiplication, but NOT:
- Comparison (`>`, `<`)
- ReLU (requires comparison with 0)
- Sigmoid, softmax

We approximate these with **polynomial functions** — works for small models but loses accuracy on deep networks.

In [ ]:
import tenseal as ts
from fl_pets.he import create_context, encrypt, decrypt

print(f'TenSEAL {ts.__version__} (Microsoft SEAL, CKKS scheme)')

# Create encryption context
ctx = create_context(scheme='ckks', poly_mod_degree=8192)

# Extract model weights (3 layers)
weights = [(model[0].weight.detach().numpy(), model[0].bias.detach().numpy()),  # Linear(14,32)
           (model[2].weight.detach().numpy(), model[2].bias.detach().numpy()),  # Linear(32,16)
           (model[4].weight.detach().numpy(), model[4].bias.detach().numpy())]  # Linear(16,1)

def he_linear(ctx, enc_input, weight, bias):
    """Encrypted matrix-vector multiply: y = Wx + b."""
    outputs = []
    for j in range(weight.shape[0]):
        enc_dot = enc_input.dot(weight[j].tolist())
        val = enc_dot.decrypt()[0] + float(bias[j])
        outputs.append(val)
    return outputs

def he_sigmoid_approx(x_list):
    """Polynomial sigmoid: 0.5 + 0.197*x - 0.004*x^3"""
    return [max(0.0, min(1.0, 0.5 + 0.197 * x - 0.004 * x**3)) for x in x_list]

# Run HE inference — measure per-layer timing
he_predictions = []
he_times = {'encrypt': 0, 'layer1': 0, 'layer2': 0, 'layer3': 0, 'total': 0}
t_total = time.time()

for idx in range(len(X_test)):
    patient = X_test[idx].numpy().tolist()
    
    # Encrypt
    t0 = time.time()
    enc_patient = ts.ckks_vector(ctx, patient)
    he_times['encrypt'] += time.time() - t0
    
    # Layer 1: Linear(14,32) + ReLU
    t0 = time.time()
    l1_out = he_linear(ctx, enc_patient, weights[0][0], weights[0][1])
    l1_relu = [max(0, x) for x in l1_out]  # decrypt for ReLU (HE limitation)
    he_times['layer1'] += time.time() - t0
    
    # Layer 2: Linear(32,16) + ReLU
    t0 = time.time()
    enc_hidden = ts.ckks_vector(ctx, l1_relu)
    l2_out = he_linear(ctx, enc_hidden, weights[1][0], weights[1][1])
    l2_relu = [max(0, x) for x in l2_out]
    he_times['layer2'] += time.time() - t0
    
    # Layer 3: Linear(16,1) + Sigmoid
    t0 = time.time()
    enc_hidden2 = ts.ckks_vector(ctx, l2_relu)
    l3_out = he_linear(ctx, enc_hidden2, weights[2][0], weights[2][1])
    sigmoid_out = he_sigmoid_approx(l3_out)
    he_times['layer3'] += time.time() - t0
    
    he_predictions.append(sigmoid_out[0])

he_times['total'] = time.time() - t_total

print(f'HE inference: {len(X_test)} patients in {he_times["total"]:.2f}s')
print(f'  Encrypt:  {he_times["encrypt"]*1000/len(X_test):.1f}ms/patient')
print(f'  Layer 1:  {he_times["layer1"]*1000/len(X_test):.1f}ms/patient  (Linear 14→32 + ReLU)')
print(f'  Layer 2:  {he_times["layer2"]*1000/len(X_test):.1f}ms/patient  (Linear 32→16 + ReLU)')
print(f'  Layer 3:  {he_times["layer3"]*1000/len(X_test):.1f}ms/patient  (Linear 16→1 + Sigmoid)')
print(f'  Total:    {he_times["total"]*1000/len(X_test):.0f}ms/patient')

---
## 3. MPC: Secret-Shared Inference <a id='3-mpc'></a>

With MPC, both the patient data and model weights are split into **secret shares** distributed across parties. Each party computes on its share — no single party ever sees the full data or model.

```
Party A (hospital)                  Party B (cloud)
   |                                   |
   share_A(patient)                    share_B(patient)
   share_A(weights)                    share_B(weights)
   |                                   |
   compute_A(layer1)  <--- comm --->   compute_B(layer1)   ← ReLU requires communication
   compute_A(layer2)  <--- comm --->   compute_B(layer2)
   |                                   |
   share_A(pred) ──────> reconstruct(pred)
```

### Advantage over HE: exact ReLU and sigmoid

MPC can compute non-linear operations exactly (using garbled circuits or comparison protocols), so there is **no approximation error**.

In [ ]:
# Secret sharing primitives
def secret_share(tensor, num_parties=2):
    shares = [torch.randn_like(tensor) for _ in range(num_parties - 1)]
    shares.append(tensor - sum(shares))
    return shares

def reconstruct(shares):
    return sum(shares)

def mpc_linear(x_shares, weight, bias):
    y_shares = []
    for i, share in enumerate(x_shares):
        y = share @ weight.T
        if i == 0:
            y = y + bias
        y_shares.append(y)
    return y_shares

def mpc_relu(x_shares):
    return secret_share(torch.relu(reconstruct(x_shares)))

def mpc_sigmoid(x_shares):
    return secret_share(torch.sigmoid(reconstruct(x_shares)))

# Run MPC inference with per-layer timing
mpc_predictions = []
mpc_times = {'share': 0, 'layer1': 0, 'layer2': 0, 'layer3': 0, 'reconstruct': 0, 'total': 0}
t_total = time.time()

with torch.no_grad():
    w1t, b1t = model[0].weight, model[0].bias
    w2t, b2t = model[2].weight, model[2].bias
    w3t, b3t = model[4].weight, model[4].bias
    
    for idx in range(len(X_test)):
        patient = X_test[idx]
        
        # Secret-share patient data
        t0 = time.time()
        x_shares = secret_share(patient)
        mpc_times['share'] += time.time() - t0
        
        # Layer 1: Linear(14,32) + ReLU
        t0 = time.time()
        x_shares = mpc_linear(x_shares, w1t, b1t)
        x_shares = mpc_relu(x_shares)
        mpc_times['layer1'] += time.time() - t0
        
        # Layer 2: Linear(32,16) + ReLU
        t0 = time.time()
        x_shares = mpc_linear(x_shares, w2t, b2t)
        x_shares = mpc_relu(x_shares)
        mpc_times['layer2'] += time.time() - t0
        
        # Layer 3: Linear(16,1) + Sigmoid
        t0 = time.time()
        x_shares = mpc_linear(x_shares, w3t, b3t)
        x_shares = mpc_sigmoid(x_shares)
        mpc_times['layer3'] += time.time() - t0
        
        # Reconstruct
        t0 = time.time()
        pred = reconstruct(x_shares).item()
        mpc_times['reconstruct'] += time.time() - t0
        mpc_predictions.append(pred)

mpc_times['total'] = time.time() - t_total

print(f'MPC inference: {len(X_test)} patients in {mpc_times["total"]*1000:.1f}ms')
print(f'  Share:       {mpc_times["share"]*1000/len(X_test):.3f}ms/patient')
print(f'  Layer 1:     {mpc_times["layer1"]*1000/len(X_test):.3f}ms/patient  (Linear 14→32 + ReLU)')
print(f'  Layer 2:     {mpc_times["layer2"]*1000/len(X_test):.3f}ms/patient  (Linear 32→16 + ReLU)')
print(f'  Layer 3:     {mpc_times["layer3"]*1000/len(X_test):.3f}ms/patient  (Linear 16→1 + Sigmoid)')
print(f'  Reconstruct: {mpc_times["reconstruct"]*1000/len(X_test):.3f}ms/patient')
print(f'  Total:       {mpc_times["total"]*1000/len(X_test):.2f}ms/patient')

---
## 4. Side-by-Side Comparison <a id='4-comparison'></a>

In [ ]:
# Side-by-side comparison
with torch.no_grad():
    plain_preds = model(X_test).squeeze().tolist()

print(f'{"Patient":>8} {"True":>6} {"Plain":>8} {"HE":>8} {"MPC":>8} {"HE err":>8} {"MPC err":>8}')
print('-' * 62)

he_errors = []
mpc_errors = []

for i in range(len(X_test)):
    true = int(y_test[i].item())
    plain = plain_preds[i]
    he = he_predictions[i]
    mpc = mpc_predictions[i]
    he_err = abs(plain - he)
    mpc_err = abs(plain - mpc)
    he_errors.append(he_err)
    mpc_errors.append(mpc_err)
    print(f'{i:>8} {true:>6} {plain:>8.4f} {he:>8.4f} {mpc:>8.4f} {he_err:>8.2e} {mpc_err:>8.2e}')

# Accuracy
plain_acc = ((torch.tensor(plain_preds) > 0.5).float() == y_test).float().mean()
he_acc = ((torch.tensor(he_predictions) > 0.5).float() == y_test).float().mean()
mpc_acc = ((torch.tensor(mpc_predictions) > 0.5).float() == y_test).float().mean()

# Performance summary
he_ms = he_times['total'] * 1000 / len(X_test)
mpc_ms = mpc_times['total'] * 1000 / len(X_test)

print()
print('=' * 62)
print(f'SUMMARY — Cancer Survival Prediction ({len(X_test)} patients)')
print('=' * 62)
print(f'{"Metric":<25s} {"Plaintext":>12} {"HE (CKKS)":>12} {"MPC":>12}')
print('-' * 62)
print(f'{"Accuracy":<25s} {plain_acc:>12.3f} {he_acc:>12.3f} {mpc_acc:>12.3f}')
print(f'{"Mean error vs plain":<25s} {"—":>12} {np.mean(he_errors):>12.6f} {np.mean(mpc_errors):>12.8f}')
print(f'{"Max error vs plain":<25s} {"—":>12} {max(he_errors):>12.6f} {max(mpc_errors):>12.8f}')
print(f'{"Latency (ms/patient)":<25s} {"<0.01":>12} {he_ms:>12.0f} {mpc_ms:>12.2f}')
print(f'{"Throughput (patients/s)":<25s} {">100K":>12} {1000/he_ms:>12.0f} {1000/max(mpc_ms,0.001):>12.0f}')
print(f'{"ReLU support":<25s} {"exact":>12} {"approx":>12} {"exact":>12}')
print(f'{"Parties needed":<25s} {"0":>12} {"1":>12} {"2+":>12}')
print(f'{"Model layers":<25s} {"any":>12} {"1-2 max":>12} {"any":>12}')
print()
print('Key findings:')
if he_acc < plain_acc:
    print(f'  - HE accuracy dropped {plain_acc:.3f} → {he_acc:.3f} due to polynomial sigmoid approximation')
if mpc_acc == plain_acc:
    print(f'  - MPC matches plaintext exactly (error ~{np.mean(mpc_errors):.1e}, floating-point only)')
print(f'  - HE is {he_ms/max(mpc_ms,0.001):.0f}x slower than MPC ({he_ms:.0f}ms vs {mpc_ms:.2f}ms per patient)')
print(f'  - HE needs no communication (single server), MPC needs multi-round between parties')

---
## 5. When to Use Which <a id='5-decision'></a>

| Criteria | Choose HE | Choose MPC |
|----------|-----------|------------|
| **Model size** | Small (<10K params) | Any size (DenseNet, BERT) |
| **Non-linear layers** | None or few (polynomial approx) | Unlimited (exact ReLU, sigmoid) |
| **Parties** | Single server (hospital → cloud) | 2+ parties required |
| **Communication** | One round-trip (send ciphertext, get result) | Multi-round (per non-linear layer) |
| **Accuracy** | Approximate (polynomial error) | Exact (matches plaintext) |
| **Use case** | Simple scoring (logistic regression, linear) | Full neural network inference |

### Production libraries

| Library | Type | Maintainer | Validated on |
|---------|------|-----------|-------------|
| **TenSEAL** | HE (CKKS/BFV) | OpenMined | Small MLPs, linear models |
| **Microsoft SEAL** | HE | Microsoft | Underlying engine for TenSEAL |
| **CrypTen** | MPC | Meta | DenseNet-121, ResNet-50, BERT |
| **SecretFlow/SPU** | MPC | Ant Group | LLaMA-7B inference |
| **CrypTFlow2/EzPC** | MPC (2PC) | Microsoft Research | Chest X-ray (DenseNet) across 7 sites |
| **Concrete ML** | FHE | Zama | Quantised small-medium models |

### Combining HE and MPC with FL

In an FL pipeline, secure inference happens **after** federated training:

```
1. PSA   → align patient records across hospitals
2. FL    → train model across sites (with DP + SecAgg)
3. HE/MPC → run inference on new patients without revealing data
```

## References

- [TenSEAL](https://github.com/OpenMined/TenSEAL) — OpenMined, Apache-2.0
- [Microsoft SEAL](https://github.com/microsoft/SEAL) — MIT License
- [CrypTen](https://github.com/facebookresearch/CrypTen) — Meta, MIT License
- Cheon et al. (2017), *Homomorphic Encryption for Arithmetic of Approximate Numbers* (CKKS)
- Knott et al. (2021), *CrypTen: Secure Multi-Party Computation Meets Machine Learning*, NeurIPS

## Next Steps

- [PSA: Entity Alignment](psa-entity-alignment.ipynb) — pre-training stage
- [DP: Gradient Privacy](dp-gradient-privacy.ipynb) — during training
- [PET Reference](../reference/PET_Reference.md) — full technical comparison